In [31]:
# Reference: https://www.youtube.com/watch?v=tHL5STNJKag
%pip install ipywidgets
from gettext import install
import glob
import sys
import os
import pip
import roboflow
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import timm
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from PIL import Image

# Print versions for verification
print('System Version:', sys.version)
print('PyTorch version:', torch.__version__)
print('Torchvision version:', torchvision.__version__)
print('Numpy version:', np.__version__)
print('Pandas version:', pd.__version__)

Note: you may need to restart the kernel to use updated packages.
System Version: 3.10.11 (v3.10.11:7d4cc5aa85, Apr  4 2023, 19:05:19) [Clang 13.0.0 (clang-1300.0.29.30)]
PyTorch version: 2.14.1
Torchvision version: 0.29.2
Numpy version: 2.2.6
Pandas version: 2.3.3


In [32]:
# Import configuration settings
try:
    from apiconfig import ROBOFLOW_API_KEY, WORKSPACE_NAME, PROJECT_NAME
except ImportError:
    raise ImportError(
        "Could not import from 'apiconfig.py'. "
        "Ensure 'apiconfig.py' exists in your workspace directory."
    )

# Validate API key and configuration
if not ROBOFLOW_API_KEY or ROBOFLOW_API_KEY == "None" or ROBOFLOW_API_KEY == "your_roboflow_api_key_here":
    raise ValueError(
        "Invalid ROBOFLOW_API_KEY detected! "
        "Please open 'apiconfig.py' and set ROBOFLOW_API_KEY to your actual string key "
        "(e.g., ROBOFLOW_API_KEY = 'xyz123...'), not None or 'None'."
    )

from roboflow import Roboflow

print(f"Connecting to Roboflow workspace '{WORKSPACE_NAME}', project '{PROJECT_NAME}'...")
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_NAME).project(PROJECT_NAME)

# Dataset 3 from the MMA3001 brief uses version 4.
dataset_location = os.path.join(
    os.getcwd(),
    "Pork-Rasher-Error(Packaging)-4",
)
dataset_is_downloaded = (
    os.path.isfile(os.path.join(dataset_location, "data.yaml"))
    and os.path.isdir(os.path.join(dataset_location, "train", "images"))
    and os.path.isdir(os.path.join(dataset_location, "valid", "images"))
)
if dataset_is_downloaded:
    print("Using existing dataset at:", dataset_location)
else:
    dataset = project.version(4).download("yolov7", location=dataset_location)
    dataset_location = dataset.location
    print("Dataset downloaded successfully to:", dataset_location)


Connecting to Roboflow workspace 'hello-2aqe0', project 'pork-rasher-error-packaging'...
loading Roboflow workspace...
loading Roboflow project...
Using existing dataset at: /Users/blakemacpherson/Documents/MMA3001/VS Code/Code/Pork-Rasher-Error(Packaging)-4


In [33]:
# Reference: https://www.youtube.com/watch?v=tHL5STNJKag
class PorkRasherPackagingDataset(Dataset):
    def __init__(self, images_dir, labels_dir, transform=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transform = transform
        self.image_paths = []
        for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG'):
            self.image_paths.extend(glob.glob(os.path.join(self.images_dir, ext)))
        self.image_paths.sort()
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label_path = os.path.join(
            self.labels_dir,
            os.path.splitext(os.path.basename(image_path))[0] + ".txt",
        )
        with open(label_path, "r", encoding="utf-8") as label_file:
            label_rows = [
                [float(value) for value in line.split()]
                for line in label_file
                if line.strip()
            ]
        label = torch.tensor(label_rows, dtype=torch.float32).reshape(-1, 5)
        return image, label

#images already resized to 200x200 in the dataset so i can test at a reasonable time
transform = transforms.Compose([ transforms.Resize((200, 200)), transforms.ToTensor(), ])

trainingdataset = PorkRasherPackagingDataset(
    images_dir=os.path.join(dataset_location, "train", "images"),
    labels_dir=os.path.join(dataset_location, "train", "labels"),
    transform=transform
) 


print((trainingdataset.__getitem__(4)))



(tensor([[[0.1961, 0.2392, 0.2118,  ..., 0.2824, 0.3294, 0.3843],
         [0.1804, 0.2078, 0.1961,  ..., 0.3294, 0.3373, 0.3529],
         [0.1529, 0.1882, 0.1922,  ..., 0.5176, 0.5098, 0.4745],
         ...,
         [0.1569, 0.0902, 0.0667,  ..., 0.9725, 0.8745, 0.6902],
         [0.1647, 0.1647, 0.1569,  ..., 0.9451, 0.7725, 0.6275],
         [0.1725, 0.1961, 0.2078,  ..., 0.8667, 0.7020, 0.6196]],

        [[0.1490, 0.2353, 0.1882,  ..., 0.2863, 0.3412, 0.3882],
         [0.1804, 0.2510, 0.2157,  ..., 0.3176, 0.3333, 0.3412],
         [0.1725, 0.2431, 0.2235,  ..., 0.5176, 0.5216, 0.4941],
         ...,
         [0.1765, 0.0902, 0.0549,  ..., 0.7216, 0.5412, 0.3255],
         [0.1961, 0.1725, 0.1569,  ..., 0.6627, 0.4078, 0.2431],
         [0.2000, 0.2039, 0.2078,  ..., 0.5686, 0.3373, 0.2471]],

        [[0.1686, 0.2039, 0.1882,  ..., 0.2784, 0.3176, 0.3608],
         [0.2078, 0.2275, 0.2118,  ..., 0.2980, 0.3137, 0.3216],
         [0.2196, 0.2392, 0.2235,  ..., 0.5137, 0.5294, 0

In [34]:
#Ref: Tashvi

def collate_fn(batch):
    """
    Custom collate function to handle batches of images and labels.

    """
    images, labels = zip(*batch)
    return torch.stack(images), list(labels)


In [35]:
trainingdataset = PorkRasherPackagingDataset(
    images_dir=os.path.join(dataset_location, "train", "images"),
    labels_dir=os.path.join(dataset_location, "train", "labels"),
    transform=transform
) 

validationdataset = PorkRasherPackagingDataset(
    images_dir=os.path.join(dataset_location, "valid", "images"),
    labels_dir=os.path.join(dataset_location, "valid", "labels"),
    transform=transform
)

trainingdataloader = DataLoader(
    trainingdataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

validationdataloader = DataLoader(
    validationdataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)
images, labels = next(iter(trainingdataloader))
print(f"First batch contains {len(labels)} label tensors:")
print(images.shape, labels[0].shape)  # Print the shape of the first label tensor
print(f"First label tensor:\n{labels[0]}")  # Print the first label tensor

First batch contains 32 label tensors:
torch.Size([32, 3, 200, 200]) torch.Size([1, 5])
First label tensor:
tensor([[1.0000, 0.8827, 0.7624, 0.0586, 0.0926]])


In [36]:
"""
# Find the distinct class IDs in the training labels
class_ids = set()
for label_path in glob.glob(os.path.join(trainingdataset.labels_dir, "*.txt")):
    with open(label_path, "r", encoding="utf-8") as label_file:
        for line in label_file:
            if line.strip():
                class_ids.add(int(float(line.split()[0])))

class_ids = (class_ids)
if not class_ids:
    raise ValueError("No class IDs found in the training labels.")

num_classes = len(class_ids)
class_to_index = {class_id: index for index, class_id in enumerate(class_ids)}
print("Class IDs:", class_ids, "| Model outputs:", num_classes)


class PorkRasherClassifer(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.base_model = timm.create_model("efficientnet_b0", pretrained=True)
        self.features = nn.Sequential(*list(self.base_model.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1280, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = PorkRasherClassifer(num_classes=num_classes)
"""

'\n# Find the distinct class IDs in the training labels\nclass_ids = set()\nfor label_path in glob.glob(os.path.join(trainingdataset.labels_dir, "*.txt")):\n    with open(label_path, "r", encoding="utf-8") as label_file:\n        for line in label_file:\n            if line.strip():\n                class_ids.add(int(float(line.split()[0])))\n\nclass_ids = (class_ids)\nif not class_ids:\n    raise ValueError("No class IDs found in the training labels.")\n\nnum_classes = len(class_ids)\nclass_to_index = {class_id: index for index, class_id in enumerate(class_ids)}\nprint("Class IDs:", class_ids, "| Model outputs:", num_classes)\n\n\nclass PorkRasherClassifer(nn.Module):\n    def __init__(self, num_classes):\n        super().__init__()\n        self.base_model = timm.create_model("efficientnet_b0", pretrained=True)\n        self.features = nn.Sequential(*list(self.base_model.children())[:-1])\n        self.classifier = nn.Sequential(\n            nn.Flatten(),\n            nn.Linear(1280

In [37]:
# Use a reserved ID for images with no defects; YOLO class IDs are nonnegative.
NO_DEFECT_CLASS = -1

class_ids = set()
for label_path in glob.glob(os.path.join(trainingdataset.labels_dir, "*.txt")):
    with open(label_path, "r", encoding="utf-8") as label_file:
        for line in label_file:
            if line.strip():
                class_ids.add(int(float(line.split()[0])))

if not class_ids:
    raise ValueError("No defect class IDs found in the training labels.")

class_ids = sorted(class_ids)
class_to_index = {class_id: index for index, class_id in enumerate(class_ids)}
class_to_index[NO_DEFECT_CLASS] = len(class_to_index)
num_classes = len(class_to_index)

print("Class mapping (including no defects):", class_to_index)

print("Class IDs:", class_ids, "| Model outputs:", num_classes)


class PorkRasherClassifer(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.base_model = timm.create_model("efficientnet_b0", pretrained=True)
        self.features = nn.Sequential(*list(self.base_model.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1280, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = PorkRasherClassifer(num_classes=num_classes)

Class mapping (including no defects): {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, -1: 5}
Class IDs: [0, 1, 2, 3, 4] | Model outputs: 6


# Use a reserved ID for images with no defects; YOLO class IDs are nonnegative.
NO_DEFECT_CLASS = -1

class_ids = set()
for label_path in glob.glob(os.path.join(trainingdataset.labels_dir, "*.txt")):
    with open(label_path, "r", encoding="utf-8") as label_file:
        for line in label_file:
            if line.strip():
                class_ids.add(int(float(line.split()[0])))

if not class_ids:
    raise ValueError("No defect class IDs found in the training labels.")

class_ids = sorted(class_ids)
class_to_index = {class_id: index for index, class_id in enumerate(class_ids)}
class_to_index[NO_DEFECT_CLASS] = len(class_to_index)
num_classes = len(class_to_index)

print("Class mapping (including no defects):", class_to_index)

print("Class IDs:", class_ids, "| Model outputs:", num_classes)


class PorkRasherClassifer(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.base_model = timm.create_model("efficientnet_b0", pretrained=True)
        self.features = nn.Sequential(*list(self.base_model.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1280, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = PorkRasherClassifer(num_classes=num_classes)

In [38]:
"""
example_out = model(images)
example_out.shape # [batch_size, num_classes]
"""

'\nexample_out = model(images)\nexample_out.shape # [batch_size, num_classes]\n'

In [39]:
# Loss function
criterion = nn.CrossEntropyLoss()
# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [40]:
valid_indices = []
targets = []

for i, boxes in enumerate(labels):
    if boxes.numel() > 0:
        class_id = int(boxes[0, 0].item())
        valid_indices.append(i)
        targets.append(class_to_index[class_id])

if not valid_indices:
    raise ValueError("This batch contains no labeled images.")


In [41]:
"""
target_tensor = torch.tensor(targets, dtype=torch.long, device=example_out.device)
loss = criterion(example_out[valid_indices], target_tensor)


print(example_out.shape, len(labels))
loss
print(example_out.shape, len(labels), [label.shape for label in labels])
"""

'\ntarget_tensor = torch.tensor(targets, dtype=torch.long, device=example_out.device)\nloss = criterion(example_out[valid_indices], target_tensor)\n\n\nprint(example_out.shape, len(labels))\nloss\nprint(example_out.shape, len(labels), [label.shape for label in labels])\n'

In [42]:
import time

for _ in tqdm(range(10), desc="Widget test"):
    time.sleep(0.1)

Widget test:   0%|          | 0/10 [00:00<?, ?it/s]

In [44]:
# Simple training loop
import copy


num_epochs = 5
train_losses, val_losses = [], []

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = PorkRasherClassifer(num_classes=6)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


best_val_accuracy = -1.0
best_epoch = 0
best_model_state = None

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    running_corrects = 0

    for images, label_batches in tqdm(trainingdataloader, desc="Training loop"):
        images = images.to(device)

        # Use the first annotation's class; assign empty-label images to no-defect.
        targets = [
            class_to_index[int(boxes[0, 0].item())]
            if boxes.numel() > 0
            else class_to_index[NO_DEFECT_CLASS]
            for boxes in label_batches
        ]
        targets = torch.tensor(targets, dtype=torch.long, device=device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * targets.size(0)
        _, preds = torch.max(outputs, 1)
        running_corrects += torch.sum(preds == targets.data)

    train_loss = running_loss / len(trainingdataloader.dataset)
    train_losses.append(train_loss)
    train_accuracy = running_corrects.double() / len(trainingdataloader.dataset)

    model.eval()
    running_loss = 0.0
    running_corrects = 0
    with torch.no_grad():
        for images, label_batches in tqdm(validationdataloader, desc="Validation loop"):
            images = images.to(device)
            targets = [
                class_to_index[int(boxes[0, 0].item())]
                if boxes.numel() > 0
                else class_to_index[NO_DEFECT_CLASS]
                for boxes in label_batches
            ]
            targets = torch.tensor(targets, dtype=torch.long, device=device)

            outputs = model(images)
            loss = criterion(outputs, targets)
            running_loss += loss.item() * targets.size(0)

            _, preds = torch.max(outputs, 1)
            running_corrects += torch.sum(preds == targets.data)

    val_loss = running_loss / len(validationdataloader.dataset)
    val_losses.append(val_loss)
    val_accuracy = running_corrects.double() / len(validationdataloader.dataset)

    if val_accuracy > best_val_accuracy:
                best_val_accuracy = val_accuracy
                best_epoch = epoch + 1
                best_model_state = copy.deepcopy(model.state_dict())

                torch.save(best_model_state, "best_traffic_sign_cnn.pth")

    print(
        f"Epoch {epoch + 1}/{num_epochs} - "
        f"Train loss: {train_loss:.4f}, Train accuracy: {train_accuracy*100:.4f}%, Validation loss: {val_loss:.4f}, Validation accuracy: {val_accuracy*100:.4f}%"
    )


Training loop:   0%|          | 0/105 [00:00<?, ?it/s]

Validation loop:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 1/5 - Train loss: 0.8756, Train accuracy: 68.7967%, Validation loss: 0.6446, Validation accuracy: 76.6667%


Training loop:   0%|          | 0/105 [00:00<?, ?it/s]

Validation loop:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 2/5 - Train loss: 0.7057, Train accuracy: 76.4706%, Validation loss: 0.6876, Validation accuracy: 80.8333%


... (5 outputs left out)

Epoch 4/5 - Train loss: 0.4394, Train accuracy: 85.1000%, Validation loss: 0.7770, Validation accuracy: 77.5000%


Training loop:   0%|          | 0/105 [00:00<?, ?it/s]

Validation loop:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 5/5 - Train loss: 0.3160, Train accuracy: 88.8922%, Validation loss: 0.6902, Validation accuracy: 79.1667%
